# FORESEE - Protophobic

## 1. Load Libraries 

In [ ]:
import sys, os
src_path = "../../"
sys.path.append(src_path)
import numpy as np
from src.foresee import Foresee, Utility, Model
from src.utils.utility import BREM_MASSES
from matplotlib import pyplot as plt
import src.foresee as foresee_module

In [ ]:
#Create symlink to direct production spectra.
try: os.unlink('model/direct')
except: pass
os.symlink(
    src=os.path.normpath('../../../files/direct/Protophobic'),
    dst='model/direct',
    target_is_directory=True,
)

## Specifying the Model

The phenomenology of the protophobic gauge boson $X$ can be described by the following Lagrangian

\begin{equation}
 \mathcal{L} = \frac{1}{2} m_{X}^2 X^2  - g \sum  x_f \ \bar f \gamma^\mu f X_\mu
\end{equation}

with the dark photon mass $m_{X}$ and the coupling $g$ as free parameters. The parameters $x_f$ specify the couplings to different fermions, in this case $x_u = -1/3$, $x_d=2/3$, $x_e=-1$ and $x_\nu=0$ for all three generations. For the search for the gauge boson at forward experiments we need to know i) the *production rate*, ii) the *lifetime* and iii) possibly the *decay branching fractions* of dark photons as function of those two parameters. All properties are specified in the `Model` class. We initialize it with the name of the model as argument. 

In [ ]:
energy = "14"
modelname="Protophobic"
model = Model(modelname, path="./")

# Builder parameters, matching the build.py / load_model() defaults.
nsample_2body = 2000
generators_light = ["EPOSLHC", "SIBYLL", "QGSJET"][:1]
brem_configurations = ["Brem_QRA_L1.5", "Brem_QRA_L1.0", "Brem_QRA_L2.0"][:1]

**Production:** If the gauge boson is sufficiently light, it is primarily produced in the decay of pseudoscalar mesons $\pi^0$, $\eta$ and $\eta' \to \gamma X$. The branching fractions can be found in Tab. 3 of [1801.04847](https://arxiv.org/pdf/1801.04847.pdf). We find 

\begin{align}
        &\text{BR}(\pi^0 \to X \gamma) = \frac{g^2}{(\varepsilon e)^2}\times\frac{|\mathrm{BW}_\omega(m_X) - \mathrm{BW}_\rho(m_X)|^2}{|\mathrm{BW}_\omega(m_X) + \mathrm{BW}_\rho(m_X)|^2} \times\text{BR}(\pi^0\to A'\gamma) \approx 0\\
        &\text{BR}(\eta \to X \gamma) = \frac{g^2}{(\varepsilon e)^2} \times\frac{|\mathrm{BW}_\omega(m_X) - 9\,\mathrm{BW}_\rho(m_X) + 4\,\mathrm{BW}_\phi(m_X)|^2}{|\mathrm{BW}_\omega(m_X) + 9\,\mathrm{BW}_\rho(m_X) - 2\,\mathrm{BW}_\phi(m_X)|^2}\times\text{BR}(\eta \to A' \gamma)\\
        &\text{BR}(\eta' \to X \gamma) = \frac{g^2}{(\varepsilon e)^2} \times\frac{|\mathrm{BW}_\omega(m_X) - 9\,\mathrm{BW}_\rho(m_X) - 8\,\mathrm{BW}_\phi(m_X)|^2}{|\mathrm{BW}_\omega(m_X) + 9\,\mathrm{BW}_\rho(m_X) + 4\,\mathrm{BW}_\phi(m_X)|^2}\times\text{BR}(\eta' \to A' \gamma)  \ ,
\end{align}

where $BW_{V} = (1-m_{A'}^2/m_V^2 - i \Gamma_V/m_V)^{-1}$. In the following, we model the production using `EPOSLHC`, `SIBYLL`, `QGSJET`. 

In [ ]:
def BW(self, mass, pid):
    return 1 / (1 - (mass/self.masses(pid))**2 - 1j * self.widths(pid)/self.masses(pid))

foresee_module.BW = BW

In [ ]:
model.add_production_2bodydecay(
    pid0 = "221",
    pid1 = "22",
    br = "2.*0.39 * (coupling/0.303)**2 * (1-mass**2/self.masses(pid0)**2)**3 * np.abs( (BW(self,mass,'223')  - 9*BW(self,mass,'113')  + 4*BW(self,mass,'333')) / (BW(self,mass,'223')  + 9*BW(self,mass,'113')  - 2*BW(self,mass,'333')) )**2",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body,  
)

model.add_production_2bodydecay(
    pid0 = "331",
    pid1 = "22",
    br = "2.*0.023 * (coupling/0.303)**2 * (1-mass**2/self.masses(pid0)**2)**3 * np.abs( (BW(self,mass,'223')  - 9*BW(self,mass,'113')  - 8*BW(self,mass,'333')) / (BW(self,mass,'223')  + 9*BW(self,mass,'113')  + 4*BW(self,mass,'333')) )**2",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body, 
)

Additionally, the $X$ can be produced via vector meson decays $V \to X P$, where $V = \rho, \rho^0, \omega, \phi$. The branching fractions for these processes are
\begin{align}
    &\text{BR}(\rho \to X\pi) =\frac{g^2}{(\varepsilon e)^2}\times\text{BR}(\rho\to A'\pi)\\
     &\text{BR}(\rho^0 \to X\eta) =\frac{g^2}{(\varepsilon e)^2}\times\text{BR}(\rho^0\to A'\eta)\\
    &\text{BR}(\omega \to X \pi^0) = \frac{g^2}{(\varepsilon e)^2}\times\text{BR}(\omega\to A'\pi^0)\\
    &\text{BR}(\omega \to X \eta) = \frac{g^2}{(\varepsilon e)^2}\times\text{BR}(\omega\to A'\eta)\\
    &\text{BR}(\phi \to X \eta) = \frac{4g^2}{(\varepsilon e)^2}\times\text{BR}(\phi\to A'\eta) \ . 
\end{align}
In the following, we model the production using `EPOSLHC`, `SIBYLL`, `QGSJET`. 

In [ ]:
# model.add_production_2bodydecay(
#     pid0 = "213", #rho+
#     pid1 = "211", #pi+
#     br = " (coupling/0.303)**2 * 4.5e-4 * ((self.masses('pid0')**2 - (self.masses('pid1') + mass)**2)*(self.masses('pid0')**2 - (self.masses('pid1') - mass)**2)/(self.masses('pid0')**2 - self.masses('pid1')**2)**2)**(3/2)",
#     generator = generators_light,
#     energy = energy,
#     nsample = nsample_2body, 
# )

# model.add_production_2bodydecay(
#     pid0 = "-213", #rho-
#     pid1 = "-211", #pi-
#     br = " (coupling/0.303)**2 * 4.5e-4 * ((self.masses('pid0')**2 - (self.masses('pid1') + mass)**2)*(self.masses('pid0')**2 - (self.masses('pid1') - mass)**2)/(self.masses('pid0')**2 - self.masses('pid1')**2)**2)**(3/2)",
#     generator = generators_light,
#     energy = energy,
#     nsample = nsample_2body, 
# )

model.add_production_2bodydecay(
    pid0 = "113", #rho0
    pid1 = "111", #pi0,
    label = "113_111",
    br = " (coupling/0.303)**2 * 4.7e-4 * ((self.masses('pid0')**2 - (self.masses('pid1') + mass)**2)*(self.masses('pid0')**2 - (self.masses('pid1') - mass)**2)/(self.masses('pid0')**2 - self.masses('pid1')**2)**2)**(3/2)",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body, 
)


model.add_production_2bodydecay(
    pid0 = "113", #rho0
    pid1 = "221", #eta
    label = "113_221",
    br = " (coupling/0.303)**2 * 3.0e-4 * ((self.masses('pid0')**2 - (self.masses('pid1') + mass)**2)*(self.masses('pid0')**2 - (self.masses('pid1') - mass)**2)/(self.masses('pid0')**2 - self.masses('pid1')**2)**2)**(3/2)",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body, 
)

model.add_production_2bodydecay(
    pid0 = "223", #omega
    pid1 = "111", #pi0
    label = "223_111",
    br = " (coupling/0.303)**2 * 8.33e-2 * ((self.masses('pid0')**2 - (self.masses('pid1') + mass)**2)*(self.masses('pid0')**2 - (self.masses('pid1') - mass)**2)/(self.masses('pid0')**2 - self.masses('pid1')**2)**2)**(3/2)",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body, 
)

model.add_production_2bodydecay(
    pid0 = "223", #omega
    pid1 = "221", #eta
    label = "223_221",
    br = " (coupling/0.303)**2 * 4.5e-4 * ((self.masses('pid0')**2 - (self.masses('pid1') + mass)**2)*(self.masses('pid0')**2 - (self.masses('pid1') - mass)**2)/(self.masses('pid0')**2 - self.masses('pid1')**2)**2)**(3/2)",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body, 
)

model.add_production_2bodydecay(
    pid0 = "333", #phi
    pid1 = "221", #eta
    br = " 4. * (coupling/0.303)**2 * 1.306e-2 * ((self.masses('pid0')**2 - (self.masses('pid1') + mass)**2)*(self.masses('pid0')**2 - (self.masses('pid1') - mass)**2)/(self.masses('pid0')**2 - self.masses('pid1')**2)**2)**(3/2)",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body, 
)

We can also produce the protophobic gauge boson via it's resonant mixing with the SM vector mesons, in particular the $\omega$, $\rho$ and $\phi$ mesons. Following [arXiv:1810.01879](https://arxiv.org/pdf/1810.01879.pdf), we can write 

\begin{equation}
    \sigma(A') = \theta_V^2 \  \sigma(V)
    \quad\quad\text{with}\quad\quad
    \theta_V = \frac{x_V g}{g_V}  \frac{m_V^2}{m_{A'}^2 - m_V^2 + i m_V \Gamma_V}
\end{equation}

Here $g_\omega = 17$, $g_\rho=5$ and $g_\phi=-12.88$ and $x_\rho=x_\omega=1$ and $x_\phi=4$ (see Tab 2 [arXiv:1801.04847](https://arxiv.org/pdf/1801.04847.pdf)). We can specify this production mode using `model.add_production_mixing()`. In the following, we focus on the $\rho$, which provides the leading contribution. 

In [ ]:
# model.add_production_mixing(
#     pid = "113",
#     mixing = "1. * coupling /5. * 0.77545**2/abs(mass**2-0.77545**2+0.77545*0.147*1j)",
#     generator = generators_light,
#     energy = energy,
# )

# model.add_production_mixing(
#     pid = "223",
#     mixing = "1. * coupling /17. * 0.78265**2/abs(mass**2-0.78265**2+0.78265*0.00849*1j)",
#     generator = generators_light,
#     energy = energy,
# )

# model.add_production_mixing(
#     pid = "333",
#     mixing = "4. * coupling /12.88 * 1.019461**2/abs(mass**2-1.019461**2+1.019461*0.0042*1j)",
#     generator = generators_light,
#     energy = energy,
# )

The $U(1)_{B-L}$ gauge bosons can also be produced via dark Bremsstrahlung, so coherent radiation off a proton in processes such as $p p \to p p A'$. The spectra for LLPs have been obtained following the description in [1708.09389](https://arxiv.org/abs/1708.09389) and are provided in the `model/direct` directory.


In [ ]:
model.add_production_direct(
    label = "Brem",
    energy = energy,
    configuration = brem_configurations,
    coupling_ref=0.303,
    masses = BREM_MASSES,
)

**Decay:** The protophobic gauge boson can decay into a varity of light states. Here we use the lifetime and the decay branching fractions as calculated in [2201.01788](https://arxiv.org/abs/2201.01788) with the DeLiVeR tool. 

In [ ]:
model.set_ctau_1d(
    filename="model/ctau.txt", 
)

# Decay final states as (mode, PDG IDs) pairs (DeLiVeR, arXiv:2201.01788). 
# Each mode loads model/br/bfrac_<mode>.txt, including the charge-specific
# hadronic sub-channels (KK_c, KK_n, 4pi_c, ...).
decay_channels = [
    ("elec", [11, -11]), ("muon", [13, -13]),
    ("2pi",          [211, -211]),
    ("3pi",          [211, -211, 111]),
    ("PiGamma",      [111, 22]),
    ("EtaGamma",     [221, 22]),
    ("EtaOmega",     [221, 223]),
    ("EtaPhi",       [221, 333]),
    ("PhiPi",        [333, 111]),
    ("OmegaPion",    [223, 111]),
    ("EtaPiPi",      [221, 211, -211]),
    ("EtaPrimePiPi", [331, 211, -211]),
    ("4pi_c",        None),              # 2pi+ 2pi-
    ("4pi_n",        None),              # pi+ pi- 2pi0
    ("6pi_c",        None),              # 3pi+ 3pi-
    ("6pi_n",        None),              # 2pi+ 2pi- 2pi0
    ("KKpipi_0",     None),              # K+ pi- K- pi+
    ("KKpipi_1",     None),              # KS pi0 K+- pi-+
    ("KKpipi_2",     None),              # K+- pi0 KS pi-+
    ("KKpipi_3",     None),              # KS pi-+ K-+ pi0
    ("KK_c",         [321, -321]),       # K+ K-
    ("KK_n",         [310, 130]),        # K0bar K0 -> K_S K_L
    ("KKpi_0",       [130, 310, 111]),   # K_L K_S pi0
    ("KKpi_1",       [321, -321, 111]),  # K+ K- pi0
    ("KKpi_2",       [321, -211, 311]),  # K+- pi-+ K0 (representative)
    ("OmPiPi_c",     [223, 211, -211]),  # omega pi+ pi-
    ("OmPiPi_n",     [223, 111, 111]),   # omega pi0 pi0
    ("PhiPiPi_c",    [333, 211, -211]),  # phi pi+ pi-
    ("PhiPiPi_n",    [333, 111, 111]),   # phi pi0 pi0
    ("ppbar",        [2212, -2212]),
    ("nnbar",        [2112, -2112]),
]
decay_modes = [mode for mode, _ in decay_channels]
finalstates = [fs for _, fs in decay_channels]
filenames = ["model/br/bfrac_" + mode + ".txt" for mode in decay_modes]

model.set_br_1d(modes=decay_modes, finalstates=finalstates, filenames=filenames)

We can now initiate FORESEE with the model that we just created. 

In [ ]:
foresee = Foresee(path=src_path)
foresee.set_model(model=model)

## 3. Event Generation

In the following, we want to study one specific benchmark point with $m_{A'}=17$ MeV and $g=10^{-5}$ and export events as a HEPMC file. 

In [ ]:
mass, coupling = 0.017, 1e-5

First, we will produce the corresponding flux for this mass and a reference coupling $g_{ref}=1$. 

In [ ]:
%%time
plot=foresee.get_llp_spectrum(mass=mass, coupling=1, do_plot=True)
os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Spectrum_{modelname}.pdf", bbox_inches="tight")
plot.show()

Next, let us define the configuration of the detector (in terms of position, size and luminosity). Here we choose FASER during 2022/2023.  

In [ ]:
foresee.set_detector(
    distance=474, 
    selection="np.sqrt(x.x**2 + (x.y+0.065)**2)<.1", 
    length=1.5, 
    luminosity=60, 
    channels=None,
)

For our benchmark point, let us now look at how many particle decay inside the decay volume. We also export 1000 unweighted events as a HEPMC file. 

In [ ]:
setupnames = generators_light
modes = {key: generators_light for key in model.production.keys()}

momenta, weights, _ = foresee.write_events(
    mass = mass, 
    coupling = coupling, 
    energy = energy, 
    numberevent = 1000,
    filename = "model/events/test.hepmc", 
    return_data = True,
    weightnames=setupnames,
    modes=modes,
)

for isetup, setup in enumerate(setupnames):
    print("Expected number of events for "+setup+":", round(sum(weights[:,isetup]),3))

Let us plot the resulting energy distribution

In [ ]:
fig = plt.figure(figsize=(7,5))
ax = plt.subplot(1,1,1)
energies = [p.e for p in momenta], 
for isetup, setup in enumerate(setupnames):
    ax.hist(energies, weights=weights[:,isetup], bins=np.logspace(2,4, 20+1), histtype='step', label=setup) 
ax.set_xscale("log")
ax.set_xlim(1e2,1e4) 
ax.set_xlabel("E [GeV]") 
ax.set_ylabel("Number of Events per Bin") 
ax.legend(frameon=False, labelspacing=0, fontsize=14, loc='upper left')
os.makedirs(f"figures/{modelname}", exist_ok=True)
plt.savefig(f"figures/{modelname}/E_distribution_{modelname}.pdf", bbox_inches="tight")
plt.show()

## 3. Sensitivity Reach

In the following, we will obtain the projected sensitivity for the LLP model. For this, we first define a grid of couplings and masses, and then produce the corresponding fluxes. 

In [ ]:
masses = [m for m in BREM_MASSES if 0.01 <= m <= 2.0]
# Extra points around each production channel kinematic endpoint, from
# utility.production_thresholds(model, mass_range=(0.01, 2.05)).
thresholds = [0.120,
    0.13093, 0.13498, 0.13903, 0.22058, 0.2274, 0.22775, 0.23422, 0.2348,
    0.24184, 0.45745, 0.4716, 0.48575, 0.53143, 0.54786, 0.5643, 0.62466,
    0.64398, 0.6633, 0.92905, 0.95778, 0.98651,
]
masses = sorted(masses + thresholds)
couplings = np.logspace(-8,-3,121)

# Use cached LLP spectra: get_llp_spectrum recomputes on every call, 
# so skip any masses already saved in model/LLP_spectra/.
for mass in masses:
    if not os.path.exists(f"model/LLP_spectra/{energy}TeV_m_{mass}.txt.gz"):
        foresee.get_llp_spectrum(mass=mass, coupling=1)

We can now plot the `production rate vs mass` using the `foresee.plot_production()` function. Below the production rates, we also show the decay branching fractions of the leading visible final states.

In [ ]:
import matplotlib.colors as mcolors

colors= list(mcolors.TABLEAU_COLORS.keys())

productions=[
    {"channels": ["221"] , "color": colors[1]   , "label": r"$\eta \to \gamma A'$" , "generators": generators_light },
    {"channels": ["331"] , "color": colors[2]   , "label": r"$\eta' \to \gamma A'$" , "generators": generators_light },
    {"channels": ["113_111"] , "color": colors[3]      , "label": r"$\rho^0 \to \pi^0 A'$", "generators": generators_light},
    {"channels": ["113_221"] , "color": colors[4]      , "label": r"$\rho^0 \to \eta A'$", "generators": generators_light },
    {"channels": ["223_111"] , "color": colors[5]      , "label": r"$\omega \to \pi^0 A'$", "generators": generators_light  },
    {"channels": ["223_221"] , "color": colors[6]      , "label": r"$\omega \to \eta A'$", "generators": generators_light },
    {"channels": ["333"] , "color": colors[7]      , "label": r"$\phi \to \eta A'$", "generators": generators_light },
    {"channels": ["Brem"], "color": colors[8], "label": r"Bremsstrahlung"       , "generators": brem_configurations},
]

branchings = [
    ["elec"    , "red"        , "solid" , r"$e^+e^-$"            , 0.050, 0.6],
    ["muon"    , "orange"     , "solid" , r"$\mu^+\mu^-$"        , 0.140, 0.10],
    ["2pi"     , "blue"       , "solid" , r"$\pi^+\pi^-$"        , 0.29,  0.1],
    ["PiGamma" , "dodgerblue" , "solid" , r"$\pi^0\gamma$" , 0.55,  0.02],
    ["3pi"     , "brown"       , "solid" , r"$\pi^0\pi^+\pi^-$"   , 0.45,   0.05],
    ["KK_c"    , "green"      , "solid" , r"$K^+K^-$"      , 1.05,  0.5],
]

plot, ax, ax2 = foresee.plot_production(
    masses = masses,
    productions = productions,
    energy=energy,
    condition="logth<-3.7 and logp>2",
    xlims=[0.01, 2.0],ylims=[2e4, 4e13],
    xlabel=r"Mass [GeV]",
    ylabel=r"Production Rate $\sigma/g^2$ [pb]",
    title=r"$\theta < 0.2$ mrad and $E > 100$ GeV",
    legendloc=(1.01,1),
    fs_label=12,
    fs_label_br=9,
    ncol=3,
    branchings=branchings,
    figsize=(7,6)
)

os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Production_{modelname}.pdf", bbox_inches="tight")
plot.show()

Let us now scan over various masses and couplings, and record the resulting number of events. Note that here we again consider the FASER configuration, which we set up before.

In [ ]:
setupnames = ['EPOSLHC']
modes = None

if energy == "13.6": detectors = [["FASER_R3"  , 480, "np.sqrt(x.x**2 + x.y**2)< .1", 1.5, 250 ,  None]]
elif energy == "14": detectors = [["FASER_HL"  , 480, "np.sqrt(x.x**2 + x.y**2)< .1", 1.5, 3000,  None], 
                                  ["FASER2_HL" , 650, "-1.5<x.x<1.5 and -.5<x.y<.5" , 10 , 3000,  None]]

condition = f"np.sqrt(p**2 + mass**2) > 100"

for detector in detectors: 

    #setup detector
    dlabel, distance, selection, length, luminosity, channels  = detector

    #skip detectors already precomputed (the plot cell reads them); scan only missing ones
    if all(os.path.exists(f"model/results/{energy}TeV_{dlabel}_{label}.npy") for label in setupnames):
        continue

    foresee.set_detector(distance=distance, selection=selection, length=length, luminosity=luminosity, channels=channels)

    #get reach  
    list_nevents = {label:[] for label in setupnames}
    for mass in masses:
        couplings, _, nevents, _, _  = foresee.get_events(mass=mass, energy=energy, couplings = couplings,modes=modes,nsample=10, preselectioncuts = condition)
        for i,label in enumerate(setupnames): list_nevents[label].append(nevents.T[i])  
            
    #save results
    configuration=dlabel
    for label in setupnames: 
        result = np.array([masses,couplings,list_nevents[label]], dtype='object')
        np.save("model/results/"+energy+"TeV_"+configuration+"_"+label+".npy",result)

We can now plot the results. For this, we first specify all detector setups for which we want to show result (filename in model/results directory, label, color, linestyle, opacity alpha for filled contours, required number of events).

In [ ]:
setups = [ 
    ["13.6TeV_FASER_R3_EPOSLHC.npy"   , r"FASER (Run 3)"    , "firebrick"         ,  "solid"  , 0., 3],
    ["14TeV_FASER_HL_EPOSLHC.npy"   , r"FASER (HL-LHC)"    , "red"         ,  "dashed"  , 0., 3],
    ["14TeV_FASER2_HL_EPOSLHC.npy"   , r"FASER2 (HL-LHC)"    , "salmon"         ,  "dashed"  , 0., 3],    
]

Then we specify all the existing bounds (filename in model/bounds directory, label, label position x, label position y, label rotation)

In [ ]:
bounds = [ 
    ["bounds_FASER.txt",  "FASER"  , 2.2e-2     , 1.3e-5          , -25  ],
    ["bounds_E137_Bjorken1988as.txt"          , "E137",  0.0101, 1.01*10**-7, 0  ],
    ["bounds_CHARM_Bergsma1985qz.txt"         , "CHARM", 0.120 , 1.2*10**-7 , 0  ],
    ["bounds_Orsay_Davier1989wz.txt"          , "Orsay", 0.042 , 2.5*10**-6 , -28],
    ["bounds_E141_Riordan1987aw.txt"          , "E141",  0.011 , 1.8*10**-5 , 0  ],
    ["bounds_NA64_Banerjee2019hmi.txt"        , "NA64",  0.014 , 1.0*10**-4 , -35],
    ["bounds_KLOE_Anastasi2015qla.txt"        , "KLOE",  0.011 , 7.0*10**-4 , 0  ],
    ["bounds_BaBar_Lees2014xha.txt"           , "BaBar", 0.060 , 7.0*10**-4 , 0  ],
    ["bounds_LHCb_Aaij2019bvg_prompt.txt"     , "LHCb" , 0.220 , 2.0*10**-5 , 0  ],
    ["bounds_LHCb_Aaij2019bvg_displaced_1.txt",  None  , 0     , 0          , 0  ],
    ["bounds_LHCb_Aaij2019bvg_displaced_2.txt",  None  , 0     , 0          , 0  ],
    ["bounds_LHCb_Aaij2019bvg_displaced_3.txt",  None  , 0     , 0          , 0  ],
    ["bounds_PADME.txt",  "PADME"  , 1.6e-2     , 2.2e-4          , 0  ],  
]

bounds2 = [ 
    ["bounds_Anomaly.txt",    "Anomaly",  0.07, 7.6*10**-5, 0  ],
]

We then specify other projected sensitivitities (filename in model/bounds directory, color, label, label position x, label position y, label rotation)

In [ ]:
projections = []

Finally, we can plot everything using `foresee.plot_reach()`. It returns a matplotlib instance, to which we can add further lines and which we can show or save. 

In [ ]:
plot = foresee.plot_reach(
    setups=setups,
    bounds=bounds,
    bounds2=bounds2,
    projections=[],
    title = "Protophobic Gauge Boson", 
    xlims = [0.01,2.5], 
    ylims = [2e-8,1e-3],
    xlabel=r"Gauge boson mass $m_{X}$ [GeV]", 
    ylabel=r"$g$" ,
    legendloc=(1.025,0.6),
    linewidths=2,
)

os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Reach_{modelname}.pdf", bbox_inches="tight")
plot.show()